# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset can contain multiple record sets (tables), each with fields and columns that are uniquely identified by their `@id` values.

In [ ]:
# Fetch dataset metadata in detail
from pprint import pprint

# Get available record sets (tables) and their @ids
record_sets = [rs['@id'] for rs in metadata.get('recordSet', [])]
print('Available record sets:')
for rs in metadata.get('recordSet', []):
    print(f"- {rs['@id']} : {rs.get('name', rs['@id'])}")
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            print(f"    - {field['@id']} : {field.get('name', field['@id'])}")
print()
# Preview first records in each record set using mlcroissant
for rset_id in record_sets:
    print(f"--- Records from record set {rset_id} ---")
    for i, record in enumerate(dataset.records(record_set=rset_id)):
        pprint(record)
        if i == 2:
            break
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

All entities are referenced by their `@id`.

In [ ]:
# Extract data from available record sets
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        print(df.head())
        print()

# For illustration, let's pick the first available record set
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"Example columns from '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using `@id` references.

You can filter, normalize, and group numeric fields, referencing them by `@id` as per Croissant standard.

In [ ]:
# EDA: Select numeric fields for demonstration.
# Find possible numeric fields from the first record set
main_df = dataframes[main_record_set_id]

# Try to find numeric columns (e.g., GAD-7, PHQ-9, PSQ scores)
possible_numeric_fields = [col for col in main_df.columns if any(keyword in col.lower() for keyword in ['gad', 'phq', 'psq', 'score', 'age'])]
print('Identified numeric fields:', possible_numeric_fields)

# Example - Use GAD-7 score, referenced by its @id
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    numeric_field_id = main_df.select_dtypes(include='number').columns[0]

# Set threshold for filtering (example: GAD-7 scores > 10)
threshold = 10

filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical key, e.g. level_of_education or gender by @id
possible_group_fields = [col for col in main_df.columns if any(keyword in col.lower() for keyword in ['education', 'gender', 'marital', 'village'])]
if possible_group_fields:
    group_field_id = possible_group_fields[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we plot the distribution of the selected numeric field and a group comparison.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution histogram for the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot comparing groups if grouping was possible
if possible_group_fields:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading, overview, extraction, and exploratory analysis of the FAIR² Mental Health Survey Dataset using `mlcroissant`.

- All dataset entities, record sets, fields, and columns were referenced by their Croissant `@id`.
- We filtered and normalized survey score fields, grouped by demographic factors, and visualized distributions.
- The dataset enables studies on mental health indicators and demographic associations in Kilifi County, Kenya.